In [2]:
import kagglehub
import pandas as pd
import numpy as np
import os
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns

# Download dataset
path = kagglehub.dataset_download("jekiwantaufik/west-java-2014-2024")
print("Path to dataset files:", path)

# List semua file dalam dataset
print("\nFile dalam dataset:")
for root, dirs, files in os.walk(path):
    for file in files:
        print(os.path.join(root, file))

100%|██████████| 5.39k/5.39k [00:00<00:00, 6.66MB/s]

Extracting files...
Path to dataset files: /root/.cache/kagglehub/datasets/jekiwantaufik/west-java-2014-2024/versions/1

File dalam dataset:
/root/.cache/kagglehub/datasets/jekiwantaufik/west-java-2014-2024/versions/1/tourism.csv


In [3]:
# Cari file CSV dalam folder
csv_files = [f for f in os.listdir(path) if f.endswith('.csv')]
print(f"File CSV yang ditemukan: {csv_files}")

# Baca file CSV pertama (atau sesuaikan dengan nama file)
if csv_files:
    # Contoh: baca file pertama
    file_path = os.path.join(path, csv_files[0])
    df = pd.read_csv(file_path)

    print("\nInformasi Dataset:")
    print(df.head())
    print(f"\nShape: {df.shape}")
    print("\nInfo:")
    print(df.info())
    print("\nKolom yang tersedia:")
    print(df.columns.tolist())
else:
    print("Tidak ditemukan file CSV dalam dataset")

File CSV yang ditemukan: ['tourism.csv']

Informasi Dataset:
   id  kode_provinsi nama_provinsi  kode_kabupaten_kota nama_kabupaten_kota  \
0   1             32    JAWA BARAT                 3201     KABUPATEN BOGOR   
1   2             32    JAWA BARAT                 3201     KABUPATEN BOGOR   
2   3             32    JAWA BARAT                 3202  KABUPATEN SUKABUMI   
3   4             32    JAWA BARAT                 3202  KABUPATEN SUKABUMI   
4   5             32    JAWA BARAT                 3203   KABUPATEN CIANJUR   

  jenis_wisatawan  jumlah_pengunjung satuan  tahun  
0     MANCANEGARA              47719  ORANG   2014  
1       NUSANTARA            1290897  ORANG   2014  
2     MANCANEGARA              49138  ORANG   2014  
3       NUSANTARA             443795  ORANG   2014  
4     MANCANEGARA               6421  ORANG   2014  

Shape: (592, 9)

Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 592 entries, 0 to 591
Data columns (total 9 columns):
 #   Column       

In [4]:
# Cek apakah ada kolom koordinat (contoh nama kolom umum)
lat_cols = [col for col in df.columns if 'lat' in col.lower() or 'latitude' in col.lower()]
lon_cols = [col for col in df.columns if 'lon' in col.lower() or 'longitude' in col.lower()]

print(f"Kolom Latitude: {lat_cols}")
print(f"Kolom Longitude: {lon_cols}")

if lat_cols and lon_cols:
    # Ambil kolom pertama yang cocok
    lat_col = lat_cols[0]
    lon_col = lon_cols[0]

    # Ekstrak koordinat
    coordinates = df[[lat_col, lon_col]].dropna()

    print(f"\nJumlah data koordinat: {len(coordinates)}")
    print(f"Range Latitude: {coordinates[lat_col].min()} - {coordinates[lat_col].max()}")
    print(f"Range Longitude: {coordinates[lon_col].min()} - {coordinates[lon_col].max()}")

    # Tampilkan beberapa titik
    print("\nContoh koordinat:")
    print(coordinates.head())

    # Visualisasi sebaran data
    plt.figure(figsize=(10, 8))
    plt.scatter(coordinates[lon_col], coordinates[lat_col], s=10, alpha=0.5)
    plt.xlabel('Longitude')
    plt.ylabel('Latitude')
    plt.title('Sebaran Titik Koordinat Jawa Barat (2014-2024)')
    plt.grid(True, alpha=0.3)
    plt.show()

    # Persiapan untuk DBSCAN
    X = coordinates.values

    # Normalisasi (penting karena skala latitude/longitude berbeda)
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
else:
    print("Dataset tidak memiliki kolom latitude/longitude yang jelas")
    # Tampilkan kolom yang ada untuk inspeksi manual
    print("\nKolom dataset:")
    for i, col in enumerate(df.columns):
        print(f"{i+1}. {col} - {df[col].dtype}")

Kolom Latitude: []
Kolom Longitude: []
Dataset tidak memiliki kolom latitude/longitude yang jelas

Kolom dataset:
1. id - int64
2. kode_provinsi - int64
3. nama_provinsi - object
4. kode_kabupaten_kota - int64
5. nama_kabupaten_kota - object
6. jenis_wisatawan - object
7. jumlah_pengunjung - int64
8. satuan - object
9. tahun - int64


In [11]:
from sklearn.neighbors import NearestNeighbors

def find_optimal_eps(data, k=5):
    """Menentukan eps optimal menggunakan k-distance graph"""
    neighbors = NearestNeighbors(n_neighbors=k)
    neighbors_fit = neighbors.fit(data)
    distances, indices = neighbors_fit.kneighbors(data)

    distances = np.sort(distances[:, k-1], axis=0)

    plt.figure(figsize=(10, 6))
    plt.plot(distances)
    plt.title(f'K-Distance Graph (k={k})')
    plt.xlabel('Data Points (sorted by distance)')
    plt.ylabel(f'Distance to {k}th Nearest Neighbor')
    plt.grid(True, alpha=0.3)

    # Tandai area elbow
    plt.axhline(y=distances[int(len(distances)*0.1)], color='r', linestyle='--', alpha=0.5, label='Area elbow')
    plt.axhline(y=distances[int(len(distances)*0.2)], color='r', linestyle='--', alpha=0.5)
    plt.legend()

    plt.show()

    # Rekomendasi eps (percentile ke-10 dari sorted distances)
    eps_recommended = distances[int(len(distances)*0.1)]
    print(f"Rekomendasi eps (percentile ke-10): {eps_recommended:.4f}")

    return eps_recommended

# Hanya jalankan jika ada data koordinat
if 'coordinates' in locals():
    eps_optimal = find_optimal_eps(X_scaled, k=5)

In [10]:
if 'X_scaled' in locals():
    # Terapkan DBSCAN dengan parameter yang ditentukan
    # eps: 0.1-0.3 umum untuk data yang sudah discale
    # min_samples: minimal 5-10 untuk data geospasial
    dbscan = DBSCAN(eps=0.2, min_samples=10, metric='euclidean')
    dbscan.fit(X_scaled)

    # Tambahkan label cluster ke dataframe asli
    coordinates_df = coordinates.copy()
    coordinates_df['cluster'] = dbscan.labels_
    coordinates_df['is_noise'] = coordinates_df['cluster'] == -1

    # Analisis hasil clustering
    n_clusters = len(set(dbscan.labels_)) - (1 if -1 in dbscan.labels_ else 0)
    n_noise = list(dbscan.labels_).count(-1)
    n_total = len(dbscan.labels_)

    print("=== HASIL DBSCAN CLUSTERING ===")
    print(f"Jumlah cluster ditemukan: {n_clusters}")
    print(f"Jumlah titik noise (outlier): {n_noise}")
    print(f"Persentase noise: {(n_noise/n_total)*100:.2f}%")
    print(f"Label cluster unik: {np.unique(dbscan.labels_)}")

    # Distribusi titik per cluster
    print("\nDistribusi titik per cluster:")
    cluster_dist = coordinates_df['cluster'].value_counts().sort_index()
    print(cluster_dist)

In [12]:
if 'coordinates_df' in locals():
    # Buat peta dengan warna berbeda per cluster
    plt.figure(figsize=(12, 10))

    # Pisahkan noise dan clusters
    noise_points = coordinates_df[coordinates_df['is_noise']]
    cluster_points = coordinates_df[~coordinates_df['is_noise']]

    # Plot noise (abu-abu)
    if len(noise_points) > 0:
        plt.scatter(noise_points[lon_col], noise_points[lat_col],
                   c='gray', s=20, alpha=0.3, label='Noise/Outlier')

    # Plot clusters (warna berbeda)
    if len(cluster_points) > 0:
        # Gunakan colormap untuk cluster yang berbeda
        unique_clusters = sorted(cluster_points['cluster'].unique())
        colors = plt.cm.tab20(np.linspace(0, 1, len(unique_clusters)))

        for cluster_id, color in zip(unique_clusters, colors):
            cluster_data = cluster_points[cluster_points['cluster'] == cluster_id]
            plt.scatter(cluster_data[lon_col], cluster_data[lat_col],
                       c=[color], s=50, alpha=0.7, label=f'Cluster {cluster_id}')

    plt.xlabel('Longitude', fontsize=12)
    plt.ylabel('Latitude', fontsize=12)
    plt.title('DBSCAN Clustering pada Data Koordinat Jawa Barat (2014-2024)', fontsize=14)
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    # Visualisasi ukuran cluster
    if n_clusters > 0:
        plt.figure(figsize=(10, 6))
        cluster_sizes = cluster_dist[cluster_dist.index != -1]

        if len(cluster_sizes) > 0:
            colors = plt.cm.viridis(np.linspace(0, 1, len(cluster_sizes)))
            bars = plt.bar(range(len(cluster_sizes)), cluster_sizes.values, color=colors)

            plt.title('Ukuran Setiap Cluster', fontsize=14)
            plt.xlabel('Cluster ID', fontsize=12)
            plt.ylabel('Jumlah Titik', fontsize=12)
            plt.xticks(range(len(cluster_sizes)), cluster_sizes.index)

            # Tambahkan label jumlah di atas bar
            for bar in bars:
                height = bar.get_height()
                plt.text(bar.get_x() + bar.get_width()/2., height,
                        f'{int(height)}', ha='center', va='bottom')

            plt.grid(True, alpha=0.3, axis='y')
            plt.tight_layout()
            plt.show()